# RAM Antivirus — Member 1: Identity & Access Attack Detection
### **Assigned Scope:** C1 Brute Force | C2 Account Takeover (ATO) | C3 IAM Privilege Escalation | C16 Defense Tampering
**Author:** Member 1 — Identity & Platform Security  
**Supporting Engines:** `backend/engines/auth_engine.py`, `ciem_engine.py`, `detectors/`

---
## Overview & Objectives
Member 1 protects the cloud identity perimeter. In this notebook, we analyze:
1. **Authentication Failure Bursts (C1 Brute Force)**
2. **Geographic / IP Fast-Flip Transitions (C2 Account Takeover)**
3. **High-Privilege Policy Attachments (C3 IAM Privilege Escalation)**
4. **Security Telemetry Tampering (C16 Defense Tampering)**


In [1]:
# %pip install -q pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("[OK] Member 1 workspace ready!")


[OK] Member 1 workspace ready!


---
## 1. C1: Brute Force Detection Logic & Failure Thresholds
- **Detection Rule:** $\ge 5$ consecutive authentication failures from the same source IP / user within a 120-second sliding window.


In [2]:
# Simulated Authentication Logs
auth_events = pd.DataFrame([
    {"timestamp": "10:00:01", "user": "admin", "src_ip": "192.168.1.50", "status": "failure"},
    {"timestamp": "10:00:15", "user": "admin", "src_ip": "192.168.1.50", "status": "failure"},
    {"timestamp": "10:00:32", "user": "admin", "src_ip": "192.168.1.50", "status": "failure"},
    {"timestamp": "10:00:48", "user": "admin", "src_ip": "192.168.1.50", "status": "failure"},
    {"timestamp": "10:01:05", "user": "admin", "src_ip": "192.168.1.50", "status": "failure"},  # Threshold met!
    {"timestamp": "10:01:20", "user": "alice", "src_ip": "10.0.0.12", "status": "success"},
])

# Sliding window aggregation
fail_counts = auth_events[auth_events["status"] == "failure"].groupby("src_ip").size()
print("Authentication Failures per IP:")
print(fail_counts)

# Threshold evaluation
BRUTE_FORCE_THRESHOLD = 5
alerts = fail_counts[fail_counts >= BRUTE_FORCE_THRESHOLD]
print(f"\n[ALERT] Brute Force Detected from IPs: {list(alerts.index)}")


Authentication Failures per IP:
src_ip
192.168.1.50    5
dtype: int64

[ALERT] Brute Force Detected from IPs: ['192.168.1.50']


---
## 2. C2: Account Takeover (ATO) Fast IP Transition Logic
- **Detection Rule:** Login from disparate geographic IPs within a short time delta, immediately followed by password or MFA modification (`ChangePassword`, `DeactivateMFA`).


In [3]:
ato_events = pd.DataFrame([
    {"user": "dev1", "action": "ConsoleLogin", "src_ip": "103.21.54.10", "city": "Mumbai", "time_min": 0},
    {"user": "dev1", "action": "ConsoleLogin", "src_ip": "45.33.32.156", "city": "Frankfurt", "time_min": 12}, # Impossible travel
    {"user": "dev1", "action": "DeactivateMFA", "src_ip": "45.33.32.156", "city": "Frankfurt", "time_min": 14}, # High risk action
])

print("User Event History:")
print(ato_events)

# Flag rapid city changes with credential mutation
has_mfa_drop = "DeactivateMFA" in ato_events["action"].values
distinct_cities = ato_events["city"].nunique()

if has_mfa_drop and distinct_cities > 1:
    print("\n[CRITICAL ALERT] Account Takeover (ATO) Detected! Impossible travel + MFA disabled.")


User Event History:
   user         action        src_ip       city  time_min
0  dev1   ConsoleLogin  103.21.54.10     Mumbai         0
1  dev1   ConsoleLogin  45.33.32.156  Frankfurt        12
2  dev1  DeactivateMFA  45.33.32.156  Frankfurt        14

[CRITICAL ALERT] Account Takeover (ATO) Detected! Impossible travel + MFA disabled.


---
## 3. C3: IAM Privilege Escalation Detection
- **Detection Rule:** Monitoring AWS CloudTrail API calls for high-risk IAM privilege grants (`AttachUserPolicy`, `PutUserPolicy`) with `AdministratorAccess`.


In [4]:
iam_events = pd.DataFrame([
    {"principal": "analyst2", "action": "AttachUserPolicy", "policy": "arn:aws:iam::aws:policy/AdministratorAccess", "severity": "critical"},
    {"principal": "app_user", "action": "ListBuckets", "policy": "None", "severity": "low"},
    {"principal": "intern1", "action": "PutUserPolicy", "policy": "InlineAdminPolicy", "severity": "critical"}
])

critical_iam = iam_events[iam_events["severity"] == "critical"]
print("Detected Privilege Escalation Events:")
critical_iam


Detected Privilege Escalation Events:


,principal,action,policy,severity
0,analyst2,AttachUserPolicy,arn:aws:iam::aws:policy/AdministratorAccess,critical
2,intern1,PutUserPolicy,InlineAdminPolicy,critical


---
## 4. C16: Defense Tampering Detection
- **Detection Rule:** Disabling security audit logs or CloudTrail (`StopLogging`, `DeleteTrail`, `DeleteFlowLogs`).


In [5]:
tampering_events = pd.DataFrame([
    {"service": "cloudtrail", "action": "StopLogging", "trail_name": "production-audit-trail"},
    {"service": "guardduty", "action": "DeleteDetector", "trail_name": "primary-detector"}
])

print("Defense Tampering Alerts:")
for idx, r in tampering_events.iterrows():
    print(f"[CRITICAL] Adversary impaired {r['service']} via '{r['action']}' on '{r['trail_name']}'")


Defense Tampering Alerts:
[CRITICAL] Adversary impaired cloudtrail via 'StopLogging' on 'production-audit-trail'
[CRITICAL] Adversary impaired guardduty via 'DeleteDetector' on 'primary-detector'


---
## 5. Identity & Access Detection Engineering Reference
- **Q1: How is brute force detected?**  
  *Ans:* Sliding window threshold ($\ge 5$ failed logins within 120 seconds grouped by IP/username).
- **Q2: Brute force vs Account Takeover?**  
  *Ans:* Brute force is repeated guessing. Account Takeover is successful compromised login followed by fast IP transition or MFA modification.
- **Q3: What is C16 Defense Tampering?**  
  *Ans:* Adversaries stopping CloudTrail logs or deleting security alarms to evade detection.
